# LLM Training a Transformer LM

This notebook now contains complete worked solutions for the Topic 4 exercises on **Training a Transformer LM**.

Each section includes:
- a short explanation of the concept being tested
- direct answers to the written questions
- executable reference implementations
- assertions or sanity checks against PyTorch behavior

The final section trains a tiny causal Transformer language model end to end, so the notebook is runnable as a compact Google Colab LM training demo on CPU or GPU.


In [1]:
# Environment bootstrap for Google Colab and local notebooks.
# Run this cell first if the runtime is missing the assignment packages.

import sys
import subprocess

REQUIRED_PACKAGES = [
    "einops>=0.8",
    "einx>=0.4",
    "jaxtyping>=0.3",
    "numpy>=2.4",
    "psutil>=7",
    "pytest>=9.0",
    "pytest-timeout",
    "pypdf>=6.10.0",
    "regex>=2026.3.32",
    "tiktoken>=0.12.0",
    "torch~=2.11.0",
    "tqdm>=4.67",
    "wandb>=0.25",
    "ty>=0.0.26",
    "ruff>=0.15.8",
]

if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *REQUIRED_PACKAGES])
else:
    print("Local runtime detected; using the currently selected kernel environment.")


Local runtime detected; using the currently selected kernel environment.


In [2]:
# Shared imports and reference implementations for Topic 4 exercises.

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import math
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

PROJECT_ROOT = Path.cwd()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", PROJECT_ROOT)
print("Torch version:", torch.__version__)
print("Device:", DEVICE)


def print_tensor_stats(name, tensor):
    print(
        f"{name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}, "
        f"mean={tensor.float().mean().item():.4f}, std={tensor.float().std().item():.4f}"
    )


def cross_entropy_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    if logits.ndim != 2:
        raise ValueError(f"Expected logits with shape (batch, vocab), got {tuple(logits.shape)}")
    if targets.ndim != 1:
        raise ValueError(f"Expected targets with shape (batch,), got {tuple(targets.shape)}")
    if logits.shape[0] != targets.shape[0]:
        raise ValueError("Batch dimension mismatch between logits and targets.")

    log_probs = F.log_softmax(logits, dim=-1)
    target_log_probs = log_probs.gather(dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)
    return -target_log_probs.mean()


def perplexity_from_loss(loss: torch.Tensor) -> torch.Tensor:
    return torch.exp(loss)


class ReferenceAdamW(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        if lr <= 0:
            raise ValueError("lr must be positive")
        if eps <= 0:
            raise ValueError("eps must be positive")
        beta1, beta2 = betas
        if not (0 <= beta1 < 1 and 0 <= beta2 < 1):
            raise ValueError("betas must each be in [0, 1)")
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group["lr"]
            beta1, beta2 = group["betas"]
            eps = group["eps"]
            weight_decay = group["weight_decay"]

            for p in group["params"]:
                if p.grad is None:
                    continue
                grad = p.grad
                if grad.is_sparse:
                    raise RuntimeError("ReferenceAdamW does not support sparse gradients.")

                state = self.state[p]
                if len(state) == 0:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p)
                    state["exp_avg_sq"] = torch.zeros_like(p)

                exp_avg = state["exp_avg"]
                exp_avg_sq = state["exp_avg_sq"]
                state["step"] += 1
                step = state["step"]

                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                bias_correction1 = 1 - beta1**step
                bias_correction2 = 1 - beta2**step
                exp_avg_hat = exp_avg / bias_correction1
                exp_avg_sq_hat = exp_avg_sq / bias_correction2

                if weight_decay != 0:
                    p.mul_(1 - lr * weight_decay)

                denom = exp_avg_sq_hat.sqrt().add_(eps)
                p.addcdiv_(exp_avg_hat, denom, value=-lr)

        return loss


def get_lr_cosine_schedule(
    it: int,
    *,
    max_lr: float,
    min_lr: float,
    warmup_iters: int,
    cosine_cycle_iters: int,
) -> float:
    if warmup_iters < 0:
        raise ValueError("warmup_iters must be non-negative")
    if cosine_cycle_iters < warmup_iters:
        raise ValueError("cosine_cycle_iters must be >= warmup_iters")

    if warmup_iters > 0 and it < warmup_iters:
        return max_lr * it / warmup_iters
    if it <= cosine_cycle_iters:
        decay_span = max(1, cosine_cycle_iters - warmup_iters)
        progress = (it - warmup_iters) / decay_span
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return min_lr + cosine * (max_lr - min_lr)
    return min_lr


def clip_grad_norm_(parameters, max_norm: float, eps: float = 1e-6):
    params = [p for p in parameters if p.grad is not None]
    if not params:
        return torch.tensor(0.0), 1.0

    total_norm = torch.linalg.vector_norm(torch.stack([p.grad.norm(2) for p in params]), 2)
    scale = min(1.0, max_norm / (total_norm.item() + eps))
    if scale < 1.0:
        for p in params:
            p.grad.mul_(scale)
    return total_norm, scale


Project root: /Users/davidbong/Documents/LLM_Assignments
Torch version: 2.11.0
Device: cpu


## cross_entropy_loss_and_perplexity

**Assignment description**

- Study Section 4.1 **Cross-entropy loss**.
- Study Section 4.1.0.1 **Perplexity**.
- Match the behavior expected by `tests/test_nn_utils.py::test_cross_entropy`.
- Explain how logits, probabilities, negative log-likelihood, and perplexity relate.

**What this is testing**

This topic checks whether you understand what the model is actually optimizing during training. A correct Transformer forward pass is not enough if you cannot compute the right learning signal from its logits.

**Pseudocode / attack plan**

```text
input logits: (batch, vocab_size)
input targets: (batch,)
compute log_probs with log-softmax over vocab dimension
gather the log probability of the correct class for each example
negate and average across the batch
return scalar cross-entropy loss

for perplexity:
perplexity = exp(cross_entropy)
```

**Implementation notes**

- Avoid computing `softmax` and then `log` separately if you care about numerical stability.
- Perplexity is just an exponential re-expression of the average loss.


In [ ]:
# cross_entropy_loss_and_perplexity completed solution

logits = torch.tensor([
    [2.0, 0.5, -1.0],
    [0.1, 1.2, 0.3],
], dtype=torch.float32)
targets = torch.tensor([0, 1])

loss = cross_entropy_loss(logits, targets)
ref_loss = F.cross_entropy(logits, targets)
perplexity = perplexity_from_loss(loss)

print_tensor_stats("logits", logits)
print("targets:", targets.tolist())
print("loss:", loss.item())
print("PyTorch reference loss:", ref_loss.item())
print("perplexity:", perplexity.item())
assert torch.allclose(loss, ref_loss)

seq_logits = torch.randn(2, 4, 5)
seq_targets = torch.randint(0, 5, (2, 4))
flat_loss = cross_entropy_loss(seq_logits.view(-1, seq_logits.size(-1)), seq_targets.view(-1))
flat_ref = F.cross_entropy(seq_logits.view(-1, seq_logits.size(-1)), seq_targets.view(-1))
assert torch.allclose(flat_loss, flat_ref)

answer_indices = (
    "Targets are class indices because each example has exactly one correct next token. "
    "Using indices lets the loss gather one log-probability per row without materializing a large one-hot matrix."
)
answer_perplexity = (
    "Perplexity is exp(cross_entropy), so lower cross-entropy always means lower perplexity because exp(x) is strictly increasing."
)
answer_relationship = (
    "Logits are unnormalized scores, softmax turns them into probabilities, negative log-likelihood selects the correct-token "
    "probability and penalizes low confidence, and cross-entropy is the batch average of those penalties."
)

print(answer_indices)
print(answer_perplexity)
print(answer_relationship)


## sgd_optimizer

**Assignment description**

- Study Section 4.2 **The SGD Optimizer**.
- Study Section 4.2.1 **Implementing SGD in PyTorch**.
- Understand what `.grad` stores and how parameters change after one step.
- Write down the minimal SGD update rule from memory.

**What this is testing**

This topic checks whether you understand training as repeated parameter updates, not as a black-box call to `optimizer.step()`. If you understand SGD, later optimizers become easier to reason about.

**Pseudocode / attack plan**

```text
for each parameter p:
    if p.grad exists:
        p = p - learning_rate * p.grad

before each backward pass:
    zero old gradients
compute loss
call backward()
apply update
```

**Writeup guidance**

- Be explicit about why gradients must be zeroed between steps.
- Note that the optimizer does not compute gradients; autograd does.


In [ ]:
# sgd_optimizer completed solution

torch.manual_seed(0)

model = torch.nn.Linear(3, 1, bias=False)
x = torch.tensor([[1.0, 2.0, 3.0]])
target = torch.tensor([[1.5]])
lr = 0.1

prediction = model(x)
loss = ((prediction - target) ** 2).mean()
loss.backward()

weights_before = model.weight.detach().clone()
grad = model.weight.grad.detach().clone()

print("weights before:", weights_before)
print("grad:", grad)

with torch.no_grad():
    model.weight -= lr * model.weight.grad

weights_after = model.weight.detach().clone()
print("weights after:", weights_after)

accum_model = torch.nn.Linear(3, 1, bias=False)
with torch.no_grad():
    accum_model.weight.copy_(weights_before)

loss_1 = ((accum_model(x) - target) ** 2).mean()
loss_1.backward()
first_grad = accum_model.weight.grad.detach().clone()
loss_2 = ((accum_model(x) - target) ** 2).mean()
loss_2.backward()
accumulated_grad = accum_model.weight.grad.detach().clone()

print("single backward grad:", first_grad)
print("grad after forgetting zero_grad:", accumulated_grad)
assert torch.allclose(accumulated_grad, 2 * first_grad)

answer_no_grad = (
    "The parameter update must run under no_grad so autograd does not treat the optimizer step as part of the differentiable forward graph."
)
answer_zero_grad = (
    "If you forget to zero gradients, PyTorch accumulates the next backward pass into the existing .grad tensor, so the update becomes the sum of multiple steps instead of the current step alone."
)
answer_math = "Minimal SGD update rule: theta_(t+1) = theta_t - eta * grad_theta L(theta_t)."

print(answer_no_grad)
print(answer_zero_grad)
print(answer_math)


## adamw

**Assignment description**

- Implement Section 4.3 **AdamW**.
- Match `tests/test_optimizer.py::test_adamw`.
- Understand momentum, second-moment tracking, bias correction, and decoupled weight decay.

**What this is testing**

This is the main optimizer implementation exercise in Topic 4. It checks whether you can translate the algorithm into a correct stateful PyTorch optimizer class.

**Pseudocode / attack plan**

```text
for each parameter p with gradient g:
    m = beta1 * m + (1 - beta1) * g
    v = beta2 * v + (1 - beta2) * g^2
    m_hat = m / (1 - beta1^t)
    v_hat = v / (1 - beta2^t)
    p = p - lr * m_hat / (sqrt(v_hat) + eps)
    p = p - lr * weight_decay * p_old_or_decoupled_form
```

**Implementation notes**

- AdamW decouples weight decay from the gradient-based adaptive update.
- Your optimizer needs persistent state per parameter, not just per module.


In [ ]:
# adamw completed solution

grad = torch.tensor([0.2, -0.4, 0.1])
param = torch.tensor([1.0, -2.0, 0.5])
m = torch.zeros_like(param)
v = torch.zeros_like(param)
beta1, beta2 = 0.9, 0.999
lr = 1e-3
eps = 1e-8
weight_decay = 0.01
t = 1

m = beta1 * m + (1 - beta1) * grad
v = beta2 * v + (1 - beta2) * (grad ** 2)
m_hat = m / (1 - beta1 ** t)
v_hat = v / (1 - beta2 ** t)
updated = param * (1 - lr * weight_decay) - lr * m_hat / (torch.sqrt(v_hat) + eps)

print("m:", m)
print("v:", v)
print("updated first-step parameter:", updated)

param_ref = torch.nn.Parameter(torch.tensor([1.0, -2.0, 0.5]))
param_torch = torch.nn.Parameter(param_ref.detach().clone())
opt_ref = ReferenceAdamW([param_ref], lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)
opt_torch = torch.optim.AdamW([param_torch], lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)

for grad_step in [
    torch.tensor([0.2, -0.4, 0.1]),
    torch.tensor([-0.3, 0.1, 0.25]),
    torch.tensor([0.05, 0.2, -0.15]),
]:
    param_ref.grad = grad_step.clone()
    param_torch.grad = grad_step.clone()
    opt_ref.step()
    opt_torch.step()

print("ReferenceAdamW param:", param_ref.detach())
print("torch.optim.AdamW param:", param_torch.detach())
assert torch.allclose(param_ref, param_torch, atol=1e-10)

answer_bias_correction = (
    "Bias correction is needed because exp_avg and exp_avg_sq start at zero, so the raw moving averages are biased toward zero during the first few steps. "
    "Dividing by 1 - beta^t removes that startup bias and makes the effective step size correct early in training."
)
answer_decoupled_decay = (
    "AdamW applies weight decay directly to the parameters before the adaptive update, while naive L2 inside Adam mixes the decay term into the gradient moments and changes the optimizer dynamics."
)

print(answer_bias_correction)
print(answer_decoupled_decay)


## learning_rate_scheduling

**Assignment description**

- Implement Section 4.4 **Learning rate scheduling**.
- Match `tests/test_optimizer.py::test_get_lr_cosine_schedule`.
- Understand linear warmup followed by cosine decay and a final floor.

**What this is testing**

This topic checks whether you can reason about the optimizer across time, not just at one update step. The schedule shape matters for stability and final convergence.

**Pseudocode / attack plan**

```text
if it < warmup_iters:
    lr = max_lr * it / warmup_iters
elif it <= cosine_cycle_iters:
    progress = (it - warmup_iters) / (cosine_cycle_iters - warmup_iters)
    cosine = 0.5 * (1 + cos(pi * progress))
    lr = min_lr + cosine * (max_lr - min_lr)
else:
    lr = min_lr
```

**Writeup guidance**

- Sketch the curve mentally: ramp up, then smooth decay, then flat floor.
- Be careful about inclusive versus exclusive boundary conditions.


In [ ]:
# learning_rate_scheduling completed solution

max_lr = 1.0
min_lr = 0.1
warmup_iters = 7
cosine_cycle_iters = 21

lrs = [
    get_lr_cosine_schedule(
        it,
        max_lr=max_lr,
        min_lr=min_lr,
        warmup_iters=warmup_iters,
        cosine_cycle_iters=cosine_cycle_iters,
    )
    for it in range(25)
]
print(lrs)

warmup_last_iter = get_lr_cosine_schedule(6, max_lr=max_lr, min_lr=min_lr, warmup_iters=warmup_iters, cosine_cycle_iters=cosine_cycle_iters)
cosine_start = get_lr_cosine_schedule(7, max_lr=max_lr, min_lr=min_lr, warmup_iters=warmup_iters, cosine_cycle_iters=cosine_cycle_iters)
cosine_end = get_lr_cosine_schedule(21, max_lr=max_lr, min_lr=min_lr, warmup_iters=warmup_iters, cosine_cycle_iters=cosine_cycle_iters)
post_floor = get_lr_cosine_schedule(24, max_lr=max_lr, min_lr=min_lr, warmup_iters=warmup_iters, cosine_cycle_iters=cosine_cycle_iters)

print("warmup last iter (it=6):", warmup_last_iter)
print("cosine starts at max lr (it=7):", cosine_start)
print("cosine ends at min lr (it=21):", cosine_end)
print("after cosine floor (it=24):", post_floor)

assert math.isclose(cosine_start, max_lr)
assert math.isclose(cosine_end, min_lr)
assert math.isclose(post_floor, min_lr)

answer_warmup = (
    "Warmup is useful because large models start training with poorly calibrated activations and optimizer statistics. "
    "Ramping the learning rate up avoids unstable early updates before the network and Adam moments settle down."
)
answer_shape = "The schedule is piecewise: linear increase, cosine decay, then a constant minimum learning-rate floor."

print(answer_warmup)
print(answer_shape)


## gradient_clipping

**Assignment description**

- Implement Section 4.5 **Gradient clipping**.
- Match `tests/test_nn_utils.py::test_gradient_clipping`.
- Understand why clipping uses the global norm across parameters.

**What this is testing**

This topic checks whether you can stabilize training when gradients become too large. The key point is that clipping rescales gradients without changing their direction when the global norm exceeds a threshold.

**Pseudocode / attack plan**

```text
collect all parameters with non-null gradients
compute total_norm = sqrt(sum over parameters of sum(grad^2))
if total_norm <= max_norm:
    do nothing
else:
    scale = max_norm / (total_norm + tiny_constant)
    multiply every gradient by scale in-place
```

**Implementation notes**

- Frozen parameters may have `grad is None`; ignore them.
- Clipping changes magnitude, not direction, when applied uniformly.


In [ ]:
# gradient_clipping completed solution

params = [torch.nn.Parameter(torch.randn(3, 3)) for _ in range(3)]
params_ref = [torch.nn.Parameter(p.detach().clone()) for p in params]

for p, p_ref in zip(params, params_ref):
    grad = torch.randn_like(p) * 10.0
    p.grad = grad.clone()
    p_ref.grad = grad.clone()

max_norm = 1.0
old_norm, scale = clip_grad_norm_(params, max_norm=max_norm)
ref_old_norm = torch.nn.utils.clip_grad_norm_(params_ref, max_norm=max_norm)
new_total_norm = torch.sqrt(sum(torch.sum(p.grad ** 2) for p in params))

print("old norm:", old_norm.item())
print("scale:", scale)
print("new norm:", new_total_norm.item())
print("torch.nn.utils.clip_grad_norm_ old norm:", ref_old_norm.item())

for p, p_ref in zip(params, params_ref):
    assert torch.allclose(p.grad, p_ref.grad, atol=1e-6)

answer_timing = (
    "Clipping belongs after backward because gradients do not exist until autograd computes them, and it must happen before optimizer.step so the optimizer uses the clipped values instead of the exploding ones."
)
answer_global = (
    "All gradients must use the same scaling factor so the full gradient vector keeps its direction. "
    "Per-parameter scaling would distort the direction and change the optimization step, not just its size."
)

print(answer_timing)
print(answer_global)


## mini_transformer_lm_training_demo

The earlier cells implement the core training utilities in isolation. This final section uses the same loss, optimizer, learning-rate schedule, and gradient clipping in a compact causal Transformer training run.

To keep the notebook lightweight enough for Google Colab CPU, the demo uses:
- a tiny character-level vocabulary
- a short repeated corpus
- a small causal Transformer
- a brief training run that should still show loss reduction and sample generation

The training mechanics are the same ones used in a larger LLM pipeline; only the scale is reduced.


In [ ]:
# mini_transformer_lm_training_demo

corpus = "\n".join(
    [
        "transformers learn next-token prediction from large text corpora.",
        "attention lets each token mix information from earlier tokens.",
        "adamw, warmup, cosine decay, and gradient clipping stabilize training.",
        "small language models can overfit tiny corpora quickly in a notebook demo.",
    ]
    * 80
)

chars = sorted(set(corpus))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(corpus), dtype=torch.long)
split = int(0.9 * len(data))
train_data = data[:split]
val_data = data[split:]

print("vocab size:", len(chars))
print("train tokens:", len(train_data))
print("val tokens:", len(val_data))


@dataclass
class TinyLMConfig:
    vocab_size: int
    block_size: int = 48
    d_model: int = 64
    num_heads: int = 4
    num_layers: int = 2
    dropout: float = 0.0


class CausalSelfAttention(nn.Module):
    def __init__(self, config: TinyLMConfig):
        super().__init__()
        if config.d_model % config.num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")
        self.num_heads = config.num_heads
        self.head_dim = config.d_model // config.num_heads
        self.qkv_proj = nn.Linear(config.d_model, 3 * config.d_model)
        self.out_proj = nn.Linear(config.d_model, config.d_model)
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(config.block_size, config.block_size, dtype=torch.bool)),
            persistent=False,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        bsz, seq_len, d_model = x.shape
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = (q @ k.transpose(-1, -2)) / math.sqrt(self.head_dim)
        attn_scores = attn_scores.masked_fill(~self.mask[:seq_len, :seq_len], float("-inf"))
        attn = torch.softmax(attn_scores, dim=-1)
        y = attn @ v
        y = y.transpose(1, 2).contiguous().view(bsz, seq_len, d_model)
        return self.out_proj(y)


class TransformerBlock(nn.Module):
    def __init__(self, config: TinyLMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.d_model)
        self.mlp = nn.Sequential(
            nn.Linear(config.d_model, 4 * config.d_model),
            nn.GELU(),
            nn.Linear(4 * config.d_model, config.d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class TinyTransformerLM(nn.Module):
    def __init__(self, config: TinyLMConfig):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.position_embedding = nn.Embedding(config.block_size, config.d_model)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.num_layers)])
        self.ln_f = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        bsz, seq_len = idx.shape
        positions = torch.arange(seq_len, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)[None, :, :]
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = cross_entropy_loss(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


config = TinyLMConfig(vocab_size=len(chars))
model = TinyTransformerLM(config).to(DEVICE)
optimizer = ReferenceAdamW(model.parameters(), lr=3e-3, weight_decay=0.01)
train_steps = 80 if DEVICE.type == "cuda" else 40
batch_size = 16
max_grad_norm = 1.0
max_lr = 3e-3
min_lr = 3e-4
warmup_iters = max(5, train_steps // 10)
cosine_cycle_iters = train_steps


def get_batch(split_name: str):
    source = train_data if split_name == "train" else val_data
    starts = torch.randint(0, len(source) - config.block_size - 1, (batch_size,))
    x = torch.stack([source[i : i + config.block_size] for i in starts]).to(DEVICE)
    y = torch.stack([source[i + 1 : i + config.block_size + 1] for i in starts]).to(DEVICE)
    return x, y


@torch.no_grad()
def estimate_loss(split_name: str, eval_batches: int = 8):
    model.eval()
    losses = []
    for _ in range(eval_batches):
        xb, yb = get_batch(split_name)
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)


initial_train_loss = estimate_loss("train")
initial_val_loss = estimate_loss("val")
print(f"initial train loss: {initial_train_loss:.4f}")
print(f"initial val loss:   {initial_val_loss:.4f}")

for step in range(train_steps):
    lr = get_lr_cosine_schedule(
        step,
        max_lr=max_lr,
        min_lr=min_lr,
        warmup_iters=warmup_iters,
        cosine_cycle_iters=cosine_cycle_iters,
    )
    for group in optimizer.param_groups:
        group["lr"] = lr

    xb, yb = get_batch("train")
    optimizer.zero_grad(set_to_none=True)
    _, loss = model(xb, yb)
    loss.backward()
    grad_norm, grad_scale = clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
    optimizer.step()

    if step % 10 == 0 or step == train_steps - 1:
        val_loss = estimate_loss("val", eval_batches=4)
        print(
            f"step={step:03d} lr={lr:.5f} train_loss={loss.item():.4f} "
            f"val_loss={val_loss:.4f} grad_norm={grad_norm.item():.4f} scale={grad_scale:.4f}"
        )

final_train_loss = estimate_loss("train")
final_val_loss = estimate_loss("val")
print(f"final train loss: {final_train_loss:.4f}")
print(f"final val loss:   {final_val_loss:.4f}")
assert final_train_loss < initial_train_loss


@torch.no_grad()
def generate(model: TinyTransformerLM, prompt: str, max_new_tokens: int = 120, temperature: float = 0.9):
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -config.block_size :]
        logits, _ = model(idx_cond)
        next_token_logits = logits[:, -1, :] / temperature
        probs = torch.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_token], dim=1)
    return decode(idx[0].tolist())


sample = generate(model, prompt="transformers ")
print("generated sample:\n")
print(sample)
